# May 2019 hearing-rate check: including pre-2010 cases

This notebook tests whether adding cases filed before 2010 makes the dashboard's May 2019 monthly hearing rate match the supplied Stage 2 calculation. It uses Purba Medinipur · CJM · Gr Case by default.

It compares three populations using the same working-day denominator: (1) Stage 2's raw all-filing-years distinct hearing dates, (2) dashboard-style hearing dates for all cases filed on or before the dashboard observation end, and (3) the dashboard's selected filing cohort. It also reports how many May dates come from pre-2010 cases and highlights the difference between Stage 2's raw-date rules and dashboard event rules.

Run from the project folder. Requires pandas, numpy, and openpyxl. No database is used or changed.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
while not (ROOT / 'config' / 'courts.json').exists():
    if ROOT == ROOT.parent:
        raise FileNotFoundError('Run this notebook from inside the project folder.')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from app.config import SOURCE_DATA_DIR, district_configurations
from app.bootstrap import source_root
from app.data_store import CsvStore
from app.domain.context import Context
from app.ingestion.raw import import_snapshot

# Edit these to check another selection.
DISTRICT = 'Purba Medinipur'
COURT = 'CJM'
CASE_TYPE = 'Gr Case'
MONTH = '2019-05'
FILED_FROM = '2010-01-01'

district_config = next(d for d in district_configurations() if d['district'] == DISTRICT)
court_config = next(c for c in district_config['courts'] if c['id'] == COURT)
START_MONTH, END_MONTH = district_config['start'], district_config['end']
SOURCE_ROOT = source_root(SOURCE_DATA_DIR, district_config)
COURT_FOLDER = SOURCE_ROOT / court_config['folder']
store = CsvStore()
snapshot_result = import_snapshot(store, SOURCE_ROOT, district_config)
dashboard = Context(store, snapshot_result['snapshot'], COURT, START_MONTH, END_MONTH)
type_row = next(r for r in dashboard.arrivals if r['case_type'] == CASE_TYPE)
TYPE_CODE = type_row['code']
MONTH_INDEX = dashboard.months.index(MONTH)
WORKDAYS = dashboard.monthly_denominators('hearing_rate')[MONTH]
print(f'{DISTRICT} · {COURT} · {CASE_TYPE} · {MONTH}; working days = {WORKDAYS}')

## Count the same month's distinct hearing dates under each population

Stage 2 counts unique dates within each CSV and adds the per-file counts. It maps each case number (CINO) to a printed case type. The dashboard instead deduplicates across its hearing files, requires the case to have a valid filing date, only includes Civil/Criminal types, filters to the filing cohort, and excludes hearing dates after filing or after a recorded decision. The cells below show these effects rather than assuming that the pre-2010 cutoff is the only difference.

In [ ]:
def read_csv(name):
    return pd.read_csv(COURT_FOLDER / name, dtype='string', keep_default_na=False)

# Recreate Stage 2 case-type mapping for all available cases, regardless of filing year.
type_lookup = read_csv('case_type_t.csv').set_index('case_type')
case_parts = []
for filename in ('civil_t.csv', 'civil_t_a.csv'):
    part = read_csv(filename)[['cino', 'regcase_type', 'date_of_filing', 'date_of_decision']].copy()
    part['filed'] = pd.to_datetime(part['date_of_filing'], errors='coerce')
    part['decided'] = pd.to_datetime(part['date_of_decision'], errors='coerce')
    case_parts.append(part)
all_cases = pd.concat(case_parts, ignore_index=True)
all_cases['case_type'] = all_cases['regcase_type'].map(type_lookup['type_name'])
all_cases['type_flag'] = all_cases['regcase_type'].map(type_lookup['type_flag'])
all_cases = all_cases.dropna(subset=['case_type', 'filed']).copy()

# Stage 2: unique hearing dates within each source file, summed across files.
stage2_month_parts = []
for filename in ('daily_proc.csv', 'daily_proc_a.csv'):
    hearing = read_csv(filename)[['cino', 'todays_date']].copy()
    hearing = hearing.loc[hearing['todays_date'].str.match(r'^[12]\d{3}-', na=False)].copy()
    hearing = hearing.loc[hearing['todays_date'].str.slice(0, 7).eq(MONTH)]
    stage2_month_parts.append(hearing.groupby('cino')['todays_date'].nunique())
stage2_counts_by_case = pd.concat(stage2_month_parts, axis=1).fillna(0).sum(axis=1).astype(int)
stage2_counts_by_case.name = 'stage2_dates'
stage2_case_types = all_cases.drop_duplicates('cino', keep='last').set_index('cino')['case_type']
stage2_mapped = stage2_counts_by_case.to_frame().join(stage2_case_types.rename('case_type'), how='inner')
stage2_may = int(stage2_mapped.loc[stage2_mapped['case_type'].eq(CASE_TYPE), 'stage2_dates'].sum())

# Rebuild each case's May dates using dashboard event rules, then bucket by filing date.
# Dashboard resolves duplicate case IDs with the archived row taking precedence.
dashboard_cases = all_cases.drop_duplicates('cino', keep='last').copy()
dashboard_cases = dashboard_cases.loc[dashboard_cases['type_flag'].isin(['1', '2'])].copy()
dashboard_cases = dashboard_cases.loc[dashboard_cases['case_type'].eq(CASE_TYPE)].copy()
observation_end = pd.Timestamp(dashboard.last)
may_rows = []
for filename in ('daily_proc.csv', 'daily_proc_a.csv'):
    hearing = read_csv(filename)[['cino', 'todays_date']].copy()
    hearing['hearing_day'] = pd.to_datetime(hearing['todays_date'], errors='coerce')
    hearing = hearing.loc[hearing['hearing_day'].dt.strftime('%Y-%m').eq(MONTH)].copy()
    # Dashboard deduplicates these dates across files, so keep raw rows for now.
    may_rows.append(hearing[['cino', 'hearing_day']])
may_hearings = pd.concat(may_rows, ignore_index=True).drop_duplicates(['cino', 'hearing_day'])
may_hearings = may_hearings.merge(dashboard_cases[['cino', 'filed', 'decided']], on='cino', how='inner')
valid_event = may_hearings['hearing_day'].notna() & may_hearings['filed'].notna()
valid_event &= may_hearings['hearing_day'].ge(may_hearings['filed'])
valid_event &= may_hearings['hearing_day'].le(observation_end)
valid_event &= may_hearings['decided'].isna() | may_hearings['hearing_day'].le(may_hearings['decided'])
dashboard_rule_may = may_hearings.loc[valid_event].copy()
dashboard_rule_may['filing_bucket'] = np.where(
    dashboard_rule_may['filed'].lt(pd.Timestamp(FILED_FROM)), 'Before 2010', '2010 onward'
)

# Current dashboard count (filings within its configured range and events through its end).
dashboard_monthly = next(r for r in dashboard.measure('monthly_rate') if r['code'] == TYPE_CODE)
current_dashboard_rate = dashboard_monthly['values'][MONTH_INDEX]
current_dashboard_dates = int(round(current_dashboard_rate * WORKDAYS))

# Each row is a unique case-day, so count rows by filing-date bucket.
dashboard_rule_bucket = dashboard_rule_may.groupby('filing_bucket').size().to_dict()
all_years_dashboard_rules = sum(dashboard_rule_bucket.values())
since_2010_dashboard_rules = dashboard_rule_bucket.get('2010 onward', 0)
pre_2010_dashboard_rules = dashboard_rule_bucket.get('Before 2010', 0)

comparison = pd.DataFrame([
    {'Population / method': 'Stage 2 raw dates · all filing years', 'May hearing dates': stage2_may},
    {'Population / method': 'Dashboard event rules · 2010 onward', 'May hearing dates': since_2010_dashboard_rules},
    {'Population / method': 'Dashboard event rules · pre-2010 cases added', 'May hearing dates': all_years_dashboard_rules},
    {'Population / method': 'Current dashboard selected filing cohort', 'May hearing dates': current_dashboard_dates},
])
comparison['Working days'] = WORKDAYS
comparison['Hearings per working day'] = (comparison['May hearing dates'] / WORKDAYS).round(2)
comparison['Difference vs Stage 2 dates'] = comparison['May hearing dates'] - stage2_may
with pd.option_context('display.max_columns', None, 'display.width', 140):
    display(comparison)
print(f'Pre-2010 contribution under dashboard event rules: {pre_2010_dashboard_rules:,} case-days')
print(f'Stage 2 raw-date total: {stage2_may:,}; dashboard event-rule total with all filing years: {all_years_dashboard_rules:,}')

## How to interpret the result

If `Dashboard event rules · pre-2010 cases added` equals Stage 2, the filing cutoff explains the entire gap for this month and selection. If it is still lower, adding older cases does not fully reconcile the methods; the remaining gap comes from event validity/deduplication/type mapping rules (or a mismatch in the source population). The `Difference vs Stage 2 dates` column quantifies that remaining gap.

The monthly rate is `distinct case hearing dates in month ÷ configured working days in month`. The notebook uses the same configured workbook denominator for every row, so only the case/hearing population changes.